# CareLoop — Part 1: ingest, clean, enrich
Run from the repository root. The source files in `data/` are preserved unchanged.

## 1.1 Read with csv and validate types

In [ ]:
import csv, json
with open('data/appointments.csv', newline='', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    header = reader.fieldnames
print('Header:', header)
print('Rows:', len(rows))

def parse_row(row):
    converted = dict(row)
    converted['age'] = int(converted['age'])
    converted['fee_inr'] = int(converted['fee_inr'])
    converted['prior_no_shows'] = int(converted['prior_no_shows'])
    converted['distance_km'] = float(converted['distance_km'])
    return converted

accepted, rejected = [], []
for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)
print('Accepted:', len(accepted), 'Rejected:', len(rejected))
with open('rejected_rows.json', 'w', encoding='utf-8') as f:
    json.dump(rejected, f, indent=2)


## 1.2 Inspect and clean pass 1

In [ ]:
import pandas as pd
import numpy as np
raw = pd.read_csv('data/appointments.csv')
print('Shape:', raw.shape)
raw.info()
print(raw.isnull().sum())
print('Exact duplicates:', raw.duplicated().sum())
df = raw.drop_duplicates().copy()
print('After duplicate removal:', len(df))
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
print('After channel cleanup:', len(df), 'distinct labels:', df.booking_channel.nunique())
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'])
print('After date conversion:', len(df))
df = df.dropna(subset=['appt_date', 'status']).copy()
print('After missing date/status removal:', len(df))


## 1.3 Validity, gaps and outliers

In [ ]:
df.loc[(df['age'] < 1) | (df['age'] > 100), 'age'] = np.nan
print('Missing age after validity check:', df.age.isna().sum())
df['age'] = df['age'].fillna(df['age'].median()).round().astype(int)
df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median())
cap_counts = {}
for col in ['distance_km', 'fee_inr']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    cap_counts[col] = int(((df[col] < lower) | (df[col] > upper)).sum())
    df[col] = np.clip(df[col], lower, upper)
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)
print('Values capped:', cap_counts)


## 1.4 Features and NumPy summary

In [ ]:
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = (df['status'] == 'No-Show').astype(int)
X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()
print('Column means [age, distance_km, fee_inr, lead_days]:', np.mean(X, axis=0))
mask = df['lead_days'].to_numpy() > 14
print('lead_days > 14:', mask.sum(), 'share:', mask.mean())
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print('Distance |z| > 2:', (np.abs(z) > 2).sum())


## 1.5 Enrich clinics with Open-Meteo (one retry)

In [ ]:
import requests, time
clinics = pd.read_csv('data/clinics.csv')
temps = []
for _, clinic in clinics.iterrows():
    params = {'latitude': clinic.latitude, 'longitude': clinic.longitude,
              'hourly': 'temperature_2m', 'forecast_days': 1}
    temp = None
    reason = 'Unknown error'
    for attempt in range(2):
        try:
            response = requests.get('https://api.open-meteo.com/v1/forecast', params=params, timeout=10)
            if response.status_code != 200:
                reason = f'HTTP {response.status_code}'
                if attempt == 0:
                    time.sleep(1)
                    continue
                break
            payload = response.json()
            values = payload.get('hourly', {}).get('temperature_2m')
            if not values:
                reason = 'hourly.temperature_2m missing or empty'
                if attempt == 0:
                    time.sleep(1)
                    continue
                break
            temp = values[0]
            break
        except (requests.exceptions.RequestException, ValueError) as exc:
            reason = str(exc)
            if attempt == 0:
                time.sleep(1)
    if temp is None:
        print(clinic.clinic_name, reason)
    temps.append(temp)
clinics['temp_c'] = temps
clinics.to_csv('clinics_enriched.csv', index=False)
clinics


## 1.6 Export clean table and SQL load script

In [ ]:
columns = ['appointment_id','patient_id','clinic_id','booked_on','appt_date','age','gender',
           'distance_km','fee_inr','booking_channel','visit_type','reminder_sent',
           'prior_no_shows','status','lead_days','no_show']
df = df.sort_values('appointment_id').copy()
clean = df[columns].copy()
for col in ['booked_on', 'appt_date']:
    clean[col] = clean[col].dt.strftime('%Y-%m-%d')
print('Unique appointment_id:', clean.appointment_id.is_unique)
print('Missing values across export:', int(clean.isna().sum().sum()))
clean.to_csv('appointments_clean.csv', index=False)

def sql_value(value):
    if pd.isna(value):
        return 'NULL'
    if isinstance(value, (int, float, np.integer, np.floating)):
        return str(value)
    return "'" + str(value).replace("'", "''") + "'"
with open('load_data.sql', 'w', encoding='utf-8') as f:
    for _, row in clinics.iterrows():
        f.write('INSERT INTO clinics VALUES (' + ','.join(sql_value(row[c]) for c in clinics.columns) + ');\n')
    for _, row in clean.iterrows():
        f.write('INSERT INTO appointments VALUES (' + ','.join(sql_value(row[c]) for c in columns) + ');\n')
